# Sesión 03 · Fine-tuning y evaluación

Cierra el bloque de SentimentLab: afinamos un modelo con reseñas propias y lo evaluamos.

Objetivos:
1. Montar un `Trainer` con `TrainingArguments` y una métrica de `evaluate`.
2. Afinar un modelo de clasificación.
3. Evaluar con `accuracy`/`f1` y guardar el modelo.

In [ ]:
%pip install -q transformers datasets evaluate

## 1 · Preparar los datos (de la sesión 02)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

dataset = load_dataset("imdb")
# Subconjunto pequeño para ir rápido en Colab gratuito
train = dataset["train"].shuffle(seed=42).select(range(1000))
test = dataset["test"].shuffle(seed=42).select(range(200))

def tokenizar(ejemplo):
    return tokenizer(ejemplo["text"], truncation=True, padding="max_length")

train = train.map(tokenizar, batched=True)
test = test.map(tokenizar, batched=True)
print("train:", len(train), "test:", len(test))

## 2 · Modelo con cabeza de clasificación

`num_labels=2` añade una cabeza nueva (positiva/negativa) sobre el tronco preentrenado.

In [ ]:
from transformers import AutoModelForSequenceClassification

modelo = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

## 3 · Argumentos y métrica

In [ ]:
from transformers import TrainingArguments
import evaluate
import numpy as np

args = TrainingArguments(
    output_dir="./resultados",
    evaluation_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    logging_steps=50,
)

accuracy = evaluate.load("accuracy")

def computar_metricas(eval_pred):
    logits, etiquetas = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=etiquetas)

## 4 · Entrenar con `Trainer`

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=train,
    eval_dataset=test,
    compute_metrics=computar_metricas,
)

trainer.train()

In [ ]:
resultado = trainer.evaluate()
print(resultado)

## 5 · Guardar y reutilizar

In [ ]:
trainer.save_model("./sentimentlab")

# Reutilizarlo con una pipeline
from transformers import pipeline
clasificador = pipeline("sentiment-analysis", model="./sentimentlab")
print(clasificador("Me ha encantado el producto."))

## Práctica propuesta (miniproyecto)

1. Repite el entrenamiento con `num_train_epochs=3` y compara el accuracy.
2. Añade la métrica `f1` y calcula ambas en `computar_metricas`.
3. Escribe una celda final comentando: ¿mejoró al subir épocas? ¿Hay riesgo de overfitting?

> Entrega: este notebook ejecutado + celda de conclusiones.

In [ ]:
# TODO: experimenta con épocas y la métrica f1, y comenta los resultados